# Chapter 17 — Exercise Solutions## GRPO and Verifiable Rewards[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com)

### Solution 17.1: Group Size Sensitivity

In [ ]:
import numpy as np, matplotlib.pyplot as plt

def advantage_variance_by_G(p_correct, G_values, n_trials=3000):
    """Simulate advantage variance as function of group size G."""
    variances = []
    for G in G_values:
        trial_vars = []
        for _ in range(n_trials):
            rewards = np.random.binomial(1, p_correct, G).astype(float)
            if rewards.std() < 1e-8: trial_vars.append(0.0); continue
            adv = (rewards - rewards.mean()) / (rewards.std() + 1e-8)
            trial_vars.append(adv.var())
        variances.append(np.mean(trial_vars))
    return variances

G_values = [2, 4, 8, 16, 32, 64]
p_values = [0.05, 0.1, 0.3, 0.5, 0.7, 0.9]

fig, ax = plt.subplots(figsize=(11, 5))
colors = plt.cm.plasma(np.linspace(0, 0.9, len(p_values)))

for p, color in zip(p_values, colors):
    var = advantage_variance_by_G(p, G_values)
    ax.plot(G_values, var, 'o-', label=f'p={p}', color=color, lw=2, ms=6)

ax.set_xlabel('Group Size G'); ax.set_ylabel('Mean Advantage Variance')
ax.set_title('GRPO Advantage Variance vs G and Task Difficulty')
ax.legend(ncol=2); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

# Where does variance stabilise?
print("Stabilisation points (variance change < 5% from G=N to G=2N):")
for p in p_values:
    var = advantage_variance_by_G(p, G_values)
    for i in range(len(G_values)-1):
        if abs(var[i+1]-var[i]) / max(var[i],1e-9) < 0.05:
            print(f"  p={p}: stabilises at G≈{G_values[i]}")
            break

print()
print("Practical recommendation from this analysis:")
print("  p > 0.3 (moderate tasks): G=8 sufficient")
print("  p < 0.1 (hard tasks):     G=16-32 needed for stable training")
print("  p < 0.05 (cold start):    G=32+ OR add SFT warm-up first")


### Solution 17.2: Format + Correctness Reward

In [ ]:
import re
import numpy as np

def correctness_reward(response: str, correct_answer: str) -> float:
    """Extract and verify boxed answer."""
    match = re.search(r'\\boxed\{([^}]+)\}', response)
    if not match: return 0.0
    answer = match.group(1).strip()
    try:
        from sympy import sympify, simplify
        return 1.0 if simplify(sympify(answer) - sympify(correct_answer)) == 0 else 0.0
    except:
        return 1.0 if answer == str(correct_answer) else 0.0

def format_reward(response: str) -> float:
    """Check for \\boxed{} format."""
    return 1.0 if re.search(r'\\boxed\{[^}]+\}', response) else 0.0

def reasoning_reward(response: str) -> float:
    """Reward evidence of reasoning steps."""
    score = 0.0
    # Reward reasoning words
    reasoning_words = ['because', 'therefore', 'since', 'thus', 'so',
                       'first', 'then', 'next', 'finally']
    score += min(sum(0.05 for w in reasoning_words if w in response.lower()), 0.1)
    # Reward showing work (multiple sentences)
    sentences = [s.strip() for s in response.split('.') if len(s.strip()) > 10]
    score += min(len(sentences) * 0.02, 0.1)
    return score

def total_reward(response: str, correct_answer: str) -> dict:
    c = correctness_reward(response, correct_answer)
    f = format_reward(response)
    r = reasoning_reward(response)
    # Weights: correctness most important
    total = 0.7 * c + 0.2 * f + 0.1 * r
    return {'correctness': c, 'format': f, 'reasoning': r, 'total': total}

# Test cases
tests = [
    ("First, 2+3=5. Therefore \\boxed{5}", "5",  "Correct + format + reasoning"),
    ("\\boxed{5}",                          "5",  "Correct + format, no reasoning"),
    ("The answer is 5",                       "5",  "Correct, no format, no reasoning"),
    ("Since x=5, therefore \\boxed{6}",     "5",  "Wrong + format + reasoning"),
    ("I don't know",                          "5",  "All wrong"),
]

print(f"{'Description':<35} {'Correct':>8} {'Format':>7} {'Reason':>7} {'Total':>7}")
print("-" * 68)
for response, answer, desc in tests:
    r = total_reward(response, answer)
    print(f"{desc:<35} {r['correctness']:>8.1f} {r['format']:>7.1f} {r['reasoning']:>7.2f} {r['total']:>7.3f}")

print("\nKey insight: format + reasoning rewards incentivise STYLE of correct reasoning")
print("Risk: model learns format without correctness (mode collapse to template responses)")
print("Mitigation: keep correctness weight high (≥0.7)")
# YOUR TURN: At what correctness weight does the model start gaming format/reasoning
# without actually improving correctness? (Hint: simulate with corrupted rewards)


### Solution 17.3: Adaptive Reasoning Length Analysis

In [ ]:
import numpy as np, matplotlib.pyplot as plt

def p_solve_given_length(difficulty_p_per_token, length):
    """P(solve) = 1 - (1-p)^length — geometric model."""
    return 1 - (1-difficulty_p_per_token)**length

def optimal_budget(difficulty_p, target_accuracy=0.8):
    """Minimum tokens needed to achieve target accuracy."""
    if difficulty_p <= 0: return float('inf')
    return int(np.log(1-target_accuracy) / np.log(1-difficulty_p)) + 1

lengths = [100, 200, 500, 1000, 2000, 4000, 8000, 16000]

difficulty_map = {
    'GSM8K (easy)':   0.008,
    'MATH (medium)':  0.002,
    'AIME (hard)':    0.0003,
    'Research (expert)': 0.00005,
}

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

for label, p in difficulty_map.items():
    accs = [p_solve_given_length(p, L) for L in lengths]
    ax1.plot(lengths, accs, 'o-', label=label, lw=2)
    opt_l = optimal_budget(p)
    ax2.bar(label.split(' ')[0], opt_l, alpha=0.7)

ax1.set_xlabel('Reasoning Length (tokens)'); ax1.set_ylabel('P(solve)')
ax1.set_title('Accuracy vs Reasoning Budget'); ax1.legend()
ax1.set_xscale('log'); ax1.grid(True, alpha=0.3)

ax2.set_ylabel('Tokens for 80% Accuracy (log scale)')
ax2.set_title('Optimal Budget by Difficulty')
ax2.set_yscale('log'); ax2.grid(True, alpha=0.3)

plt.tight_layout(); plt.show()

print("Optimal token budget for 80% accuracy:")
for label, p in difficulty_map.items():
    opt = optimal_budget(p)
    print(f"  {label:<25}: {opt:8,d} tokens")

print()
print("This motivates adaptive compute allocation in deployed models:")
print("  Easy question → short budget (100-200 tokens)")
print("  Hard question → long budget (4000-16000 tokens)")
print("  RLVR training teaches the model WHICH questions are hard")
print("  → model learns to allocate compute proportionally to difficulty")

# YOUR TURN: Add a cost-accuracy tradeoff curve
# If each token costs $0.001, what is the ROI of solving each problem type?
# At what difficulty level does extended reasoning become cost-ineffective?
